In [1]:
import sys
#lets you access python's runtime environment
from pathlib import Path
#sys.path is a built in variable in the sys module and contains a list of directories that is seached through when you do an import
#so we are appending the src directory to that
sys.path.append(str(Path().resolve().parent / "src"))
import config

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

panel_data_path = config.PROJECT_ROOT/ "data" / "panel_data.csv"
panel_data = pd.read_csv(panel_data_path)

Train/test split must be time-aware. Entries in the train set should all predate entries in the test set.

In [2]:
year_counts = panel_data.groupby('Year').size().reset_index(name = 'NumberOfRows')
year_counts = year_counts.sort_values('Year')
year_counts['Cumulative'] = year_counts['NumberOfRows'].cumsum()
total_rows = year_counts.loc[len(year_counts)-1, 'Cumulative']
year_counts['EntriesPrecedingAndIncluding(%)'] = (year_counts['Cumulative']/total_rows) *100
year_counts

,Year,NumberOfRows,Cumulative,EntriesPrecedingAndIncluding(%)
0,2015,20521,20521,6.723501
1,2016,26591,47112,15.435778
2,2017,31939,79051,25.900273
3,2018,37641,116692,38.232972
4,2019,40916,157608,51.638692
5,2020,17673,175281,57.429074
6,2021,25873,201154,65.906105
7,2022,27153,228307,74.802515
8,2023,35947,264254,86.580192
9,2024,40959,305213,100.000000


Could take entries from 2023 and before for train set for approximately 87/13 train/test split. 

In [3]:
panel = panel_data.drop(columns = ['Name', 'WeightClassKg', 'Sex'])
train = panel[panel['Year']<=2022].copy()
validate = panel[panel['Year']==2023].copy()
test = panel[panel['Year']>2023].copy()

train_X = train.drop(columns = 'CompetesNextYear')
train_y = train['CompetesNextYear']
test_X = test.drop(columns = 'CompetesNextYear')
test_y = test['CompetesNextYear']

clf = RandomForestClassifier()
clf.fit(train_X, train_y)

preds = clf.predict(test_X)
acc = accuracy_score(test_y,preds)
f1 = f1_score(test_y, preds)

print(acc)
print(f1)

0.6313630703874606
0.5994853974906497


In [4]:
panel_data.groupby('CompetesNextYear').size().reset_index(name = 'Number')

,CompetesNextYear,Number
0,False,180574
1,True,124639


In [5]:
competes_multiple = panel_data.groupby('Name').size().reset_index(name = 'YearsCompeted')

In [6]:
len(panel_data[panel_data['NumberOfMeets']>1])/len(panel_data)

0.34208569097646563

### How many features to keep

Will take top N features where N is determined by taking top n features and seeing which value of n yields best performance on validation set. Can then use top N features to retrain classifier and evaluate performance using test set.

In [7]:
importances = clf.feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': train_X.columns,
    'Importance': importances
}).sort_values('Importance', ascending=False, ignore_index =True)

feature_importance_df

,Feature,Importance
0,TimeSinceLastPBYearEnd,9.706481e-02
1,Age,8.883542e-02
2,TimeCompetingYearEnd,8.629250e-02
3,Wilks,8.522428e-02
4,Goodlift,8.399494e-02
5,Dots,8.300512e-02
6,BestMeetOfYear,7.691663e-02
7,PercentageImprovementGradientWithinYear,7.366353e-02
8,Year,6.135299e-02
9,PercentageImprovementGradientTwoMeets,5.657564e-02


In [8]:
panel[['Wilks','Dots','Goodlift']].corr()


,Wilks,Dots,Goodlift
Wilks,1.000000,0.997941,0.992439
Dots,0.997941,1.000000,0.996222
Goodlift,0.992439,0.996222,1.000000


In [9]:
panel[['PercentageImprovementGradientWithinYear','PercentageImprovementGradientTwoMeets','PercentageImprovementGradientBetweenYears']].corr()

,PercentageImprovementGradientWithinYear,PercentageImprovementGradientTwoMeets,PercentageImprovementGradientBetweenYears
PercentageImprovementGradientWithinYear,1.000000,-0.000131,-0.002750
PercentageImprovementGradientTwoMeets,-0.000131,1.000000,0.040169
PercentageImprovementGradientBetweenYears,-0.002750,0.040169,1.000000


Wilks, Goodlift and Dots are all different formulas for bodyweight adjusted strength. Very high correlation between these so will only keep most predictive feature, found to be Dots.

In [10]:
# reduced_feats = feature_importance_df.loc[:13]
# reduced_feats = reduced_feats.drop(labels = [2,4], axis = 0).reset_index()
# reduced_feats

In [11]:
results = []

for i in range(len(feature_importance_df)):
    features = feature_importance_df.loc[:i, 'Feature'].to_list()
    columns = features + ['CompetesNextYear']

    train_n = train[columns]
    train_n_X = train_n.drop(columns = 'CompetesNextYear')
    train_n_y = train_n['CompetesNextYear']

    validate_n = validate[columns]
    validate_n_X = validate_n.drop(columns = 'CompetesNextYear')
    validate_n_y = validate_n['CompetesNextYear']

    clf_n = RandomForestClassifier(random_state=42)
    clf_n.fit(train_n_X, train_n_y)
    
    preds_n = clf_n.predict(validate_n_X)
    acc_n = accuracy_score(validate_n_y, preds_n)
    f1_n = f1_score(validate_n_y, preds_n)
    precision_n = precision_score(validate_n_y, preds_n)
    recall_n = recall_score(validate_n_y, preds_n)

    results.append({'n_features': len(features), 'features': features, 
                    'accuracy': acc_n,'f1': f1_n, 'precision_n': precision_n, 'recall_n':recall_n})

results_df = pd.DataFrame(results)

KeyboardInterrupt: 

In [ ]:
results_df